In [69]:
import pandas as pd
import os
import logging
import sys
from soda_core import configure_logging

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
    logger.addHandler(handler)


# Data quality with soda : first check 

In [70]:
import pandas as pd
import duckdb
import os
from datetime import datetime
from soda_duckdb import DuckDBDataSource
from soda_core.contracts import verify_contract_locally
from soda_core import configure_logging


In [71]:
def write_report(result, output_dir="../include/reports"):
    os.makedirs(output_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    report_path = os.path.join(output_dir, f"soda_report_{timestamp}.md")

    md_lines = [
        "# Soda Data Quality Report",
        "",
        f"**Date** : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        "",
    ]

    for contract_result in result.contract_verification_results:
        dataset_name = contract_result.contract.dataset_name
        status = contract_result.status

        logger.info(f"Dataset : {dataset_name} | Statut : {status}")

        started = getattr(contract_result, "started_timestamp", None)
        ended   = getattr(contract_result, "ended_timestamp", None)
        timing  = f"  |  **Début** : {started}  |  **Fin** : {ended}" if started else ""

        md_lines += [
            f"## Dataset : `{dataset_name}`",
            "",
            f"**Statut** : `{status}`{timing}",
            "",
        ]

        if contract_result.has_errors:
            errors = contract_result.get_errors_str()
            logger.error(f"Erreurs d'exécution : {errors}")
            md_lines += [f"> ❗ Erreurs d'exécution :\n> {errors}", ""]

        # Regrouper les check_results par colonne
        by_col = {}
        for cr in contract_result.check_results:
            col = cr.check.column_name or "Dataset"
            by_col.setdefault(col, []).append(cr)

        for col, check_results in by_col.items():
            md_lines += [
                f"### Colonne : `{col}`",
                "",
                "| | Check | Type | Statut | Seuil | Diagnostics |",
                "|--|-------|------|--------|-------|-------------|",
            ]
            for cr in check_results:
                emoticon    = cr.outcome_emoticon
                name        = cr.check.name or cr.check.type
                check_type  = cr.check.type
                outcome     = str(cr.outcome)
                threshold   = str(cr.threshold_value) if cr.threshold_value is not None else "-"
                diagnostics = cr.log_table_row_diagnostics(verbose=False) if cr.diagnostic_metric_values else "-"

                if cr.is_failed:
                    logger.error(f"  {emoticon} [{check_type}] {name} → {outcome} | seuil={threshold} | {diagnostics}")
                elif "WARNED" in outcome:
                    logger.warning(f"  {emoticon} [{check_type}] {name} → {outcome} | seuil={threshold}")
                else:
                    logger.info(f"  {emoticon} [{check_type}] {name} → {outcome}")

                md_lines.append(f"| {emoticon} | {name} | {check_type} | {outcome} | {threshold} | {diagnostics} |")

            md_lines.append("")

        md_lines += [
            "| Métrique | Valeur |",
            "|----------|--------|",
            f"| Total checks | {contract_result.number_of_checks} |",
            f"| ✅ Passés | {contract_result.number_of_checks_passed} |",
            f"| ❌ Échoués | {contract_result.number_of_checks_failed} |",
            "",
            "---",
            "",
        ]

    # Résumé global de la session
    logger.info(
        f"Session | total={result.number_of_checks} "
        f"passés={result.number_of_checks_passed} "
        f"échoués={result.number_of_checks_failed} "
        f"is_ok={result.is_ok}"
    )

    md_lines += [
        "## Résumé global de la session",
        "",
        "| Métrique | Valeur |",
        "|----------|--------|",
        f"| Total checks | {result.number_of_checks} |",
        f"| ✅ Passés | {result.number_of_checks_passed} |",
        f"| ❌ Échoués | {result.number_of_checks_failed} |",
        f"| is_passed | {result.is_passed} |",
        f"| is_failed | {result.is_failed} |",
        f"| is_warned | {result.is_warned} |",
        f"| has_errors | {result.has_errors} |",
        f"| is_ok | {result.is_ok} |",
    ]

    with open(report_path, "w", encoding="utf-8") as f:
        f.write("\n".join(md_lines))

    logger.info(f"Rapport Markdown généré : {report_path}")


In [ ]:
def check_soda(file_path, contract_file_path="../include/soda_scan/soda_rules_firstcheck.yml"):

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Le fichier {file_path} est introuvable.")
    if not os.path.exists(contract_file_path):
        raise FileNotFoundError(f"Le fichier {contract_file_path} est introuvable.")

    df = pd.read_json(file_path)

    # Connexion DuckDB en mémoire
    conn = duckdb.connect(database=":memory:")
    cursor = conn.cursor()
    cursor.register("chicago_crime", df)

    # Datasource Soda DuckDB
    duckdb_ds = DuckDBDataSource.from_existing_cursor(cursor, name="duckdb")

    # Activer le logging Soda avant chaque vérification (recommandé par la doc si appelé en boucle)
    configure_logging(verbose=False)

    # Vérification du contrat
    result = verify_contract_locally(
        data_sources=[duckdb_ds],
        contract_file_path=contract_file_path
    )

    # Logs Soda internes (debug)
    soda_logs = result.get_logs_str()
    if soda_logs:
        logger.debug(f"Logs Soda internes :\n{soda_logs}")

    # Résultat global de session
    if result.has_errors:
        logger.error(f"Erreurs de vérification :\n{result.get_errors_str()}")
    elif result.is_failed:
        logger.warning(f"Contrat échoué : {result.number_of_checks_failed} check(s) en échec.")
    elif result.is_warned:
        logger.warning("Contrat avec avertissements.")
    else:
        logger.info(f"Contrat validé : {result.number_of_checks_passed}/{result.number_of_checks} checks passés.")

    write_report(result)
    return result


In [73]:
# Charger le JSON en DataFrame
file_path = "../include/data/raw_crimes_20260310_112555.json"
check_soda(file_path)

Verifying contract 📜 ../soda_scan/soda_rules_firstcheck.yml 🤞

### Contract results for duckdb/chicago_crime
+----------------+---------------------+-------------+-----------+----------------------------+
| Column         | Check               | Threshold   | Outcome   | Diagnostics                |
+================+=====================+=============+===========+============================+
| arrest         | No missing values   | level: fail | ✅ PASSED | missing_count: 0           |
|                |                     | must be: 0  |           | missing_percent: 0.0       |
|                |                     |             |           | check_rows_tested: 20000   |
|                |                     |             |           | dataset_rows_tested: 20000 |
+----------------+---------------------+-------------+-----------+----------------------------+
| beat           | No invalid values   | level: fail | ✅ PASSED | invalid_count: 0           |
|                |           